# Splitting and Embedding Text Using LangChain (Similarity Search)


In [1]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)

True

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

with open('files/churchill_speech.txt') as f:
    churchill_speech = f.read()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=60,
    length_function=len
)

In [ ]:
chunks = text_splitter.create_documents([churchill_speech])
# print(chunks[2])
# print(chunks[10].page_content)
print(f'Now you have {len(chunks)}')

#### Embedding Cost

In [4]:
def print_embedding_cost(texts):
    import tiktoken
    enc = tiktoken.encoding_for_model('text-embedding-3-small')
    total_tokens = sum([len(enc.encode(page.page_content)) for page in texts])
    # check prices here: https://openai.com/pricing
    print(f'Total Tokens: {total_tokens}')
    print(f'Embedding Cost in USD: {total_tokens / 1000 * 0.00002:.6f}')
    
print_embedding_cost(chunks)

Total Tokens: 4820
Embedding Cost in USD: 0.000096


In [6]:
! pip install langchain-huggingface sentence-transformers

  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
  Using cached setuptools-84.2.1-py3-none-any.whl.metadata (6.6 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached annotated_doc-0.0.5-py3-none-any.whl.metadata (6.5 kB)
   ---------------------------------------- 0.0/846.4 kB ? eta -:--:--
   ---------------------------------------- 846.4/846.4 kB 7.9 MB/s  0:00:00
   ---------------------------------------- 0.0/3.9 MB ? eta -:--:--
   ------------------ --------------------- 1.8/3.9 MB 11.1 MB/s eta 0:00:01
   ---------------------------------------- 3.9/3.9 MB 11.3 MB/s  0:00:00
   ---------------------------------------- 0.0/2.9 MB ? eta -:--:--
   ------------------ --------------------- 1.3/2.9 MB 6.5 MB/s eta 0:00:01
   ---------------------------------------- 2.9/2.9 MB 6.6 MB/s  0:00:00
   ---------------------------------------- 0.0/760.7 kB ? eta -:--:--
   ---------------------------------------- 760.7/760.7 kB 6.7 MB/s  0:00:


[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


### Creating embeddings

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

d:\LEARNING_IN_PROGRESS\UdemyLangChain\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4279.01it/s]


In [9]:
vector = embeddings.embed_query(chunks[0].page_content)
print(vector)

[0.05969041958451271, 0.05481329932808876, -0.012999423779547215, 0.013887408189475536, 0.03500540927052498, 0.030245909467339516, 0.015221056528389454, -0.12963677942752838, -0.04779360070824623, -0.009122766554355621, -0.06219657510519028, -0.026451509445905685, 0.034692589193582535, 0.0876588374376297, 0.07212189584970474, 0.12327327579259872, -0.028904695063829422, 0.0011453350307419896, -0.013442184776067734, 0.005183422937989235, -0.02201656810939312, 0.11959370970726013, 0.020534513518214226, 0.04307311400771141, -0.06186554208397865, 0.03242563456296921, 0.028423190116882324, 0.04596101865172386, 0.009209569543600082, 0.06065793335437775, -0.013998707756400108, -0.045530252158641815, 0.04234815761446953, -0.024023208767175674, 0.07662779837846756, -0.0027857450768351555, 0.014203177765011787, -0.06331884115934372, -0.011359630152583122, -0.005053644999861717, -0.10448968410491943, -0.03941817209124565, 0.012686846777796745, 0.044263459742069244, 0.058851949870586395, 0.07192001

### Inserting the Embeddings into a Pinecone Index

In [18]:
# Importing the libraries and initializing the Pinecone client
import os
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
load_dotenv(find_dotenv(), override=True)
Pine_cone_api_key = os.getenv("PINECONE_API_KEY")
pc = Pinecone(api_key=Pine_cone_api_key)

In [26]:
# deleting all indexes
indexes = pc.list_indexes().names()
for i in indexes:
    print(f'Deleting index {i} ... ', end='')
    pc.delete_index(i)
    print('Done')

Deleting index churchill-speech ... Done
Deleting index rag-384 ... Done


In [27]:
# creating an index
from pinecone import ServerlessSpec
index_name = 'churchill-speech'
if index_name not in pc.list_indexes().names():
    print(f'Creating index {index_name}')
    pc.create_index(
        name=index_name,
        dimension=len(embeddings.embed_query('test')),  # 384 for all-MiniLM-L6-v2
        metric='cosine',
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        ) 
    )
    print('Index created! 😊')
else:
    print(f'Index {index_name} already exists!')

Creating index churchill-speech
Index created! 😊


In [ ]:
# Clear any vectors from earlier runs (avoids duplicates), then embed and upsert the chunks.
index = pc.Index(index_name)
if index.describe_index_stats().total_vector_count > 0:
    index.delete(delete_all=True)
vector_store = PineconeVectorStore.from_documents(chunks, embeddings, index_name=index_name)

In [ ]:
index = pc.Index(index_name)
index.describe_index_stats()

### Asking Questions (Similarity Search)

In [ ]:
query = 'Where should we fight?'
result = vector_store.similarity_search(query)
print(result)

In [ ]:
for r in result:
    print(r.page_content)
    print('-' * 50)

### Answering in Natural Language using an LLM

In [34]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Initialize Groq LLM
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.2
)

# Retrieve top 5 similar chunks
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

# The instruction lives in the prompt, NOT in the search query
prompt = ChatPromptTemplate.from_template("""Answer the question using ONLY the context below.
Give a complete answer that lists every relevant detail, and quote the speech where useful.
If the answer is not in the context, say "I don't know".

Context:
{context}

Question: {question}""")

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [35]:
query = 'Where should we fight?'
answer = chain.invoke(query)
print(answer)

**Where we should fight**

According to the speech, the fight must take place in every possible theatre:

| Location | Exact wording from the speech |
|----------|--------------------------------|
| **France** | “we shall fight in France” |
| **Seas and oceans** | “we shall fight on the seas and oceans” |
| **Air** | “we shall fight with growing confidence and growing strength in the air” |
| **Our island** | “we shall defend our Island, whatever the cost may be” |
| **Beaches** | “we shall fight on the beaches” |
| **Landing grounds** | “we shall fight on the landing grounds” |
| **Fields** | “we shall fight in the fields” |
| **Streets** | “we shall fight in the streets” |
| **Hills** | “we shall fight in the hills” |

The speech makes it clear that the fight will occur **everywhere** that is necessary: “we shall fight … wherever it is required.” It also emphasizes that the island itself must be defended, “to defend our Island home.” Thus, the answer is that we should fight in France

In [36]:
query = 'Who was the king of Belgium at that time?'
answer = chain.invoke(query)
print(answer)

The king of Belgium at that time was **King Leopold II**.  
The passage states: “**King Leopold called upon us to come to his aid**,” indicating that the monarch in question was Leopold II, who reigned over Belgium during the early years of World I.


In [37]:
query = 'What about the French Armies??'
answer = chain.invoke(query)
print(answer)

The speech describes the French Armies as being in a dire situation:

* “the whole of the **French First Army** … would be broken up in the open field.”  
* “contact was lost between the British and two out of the three corps forming the **First French Army**.”  
* “The **French Army has been weakened**.”  
* “The **Belgian, British and French Armies were almost surrounded**.”  
* “Their sole line of retreat was to a single port and to its neighboring beaches.”  

Thus, the French Armies were weakened, fragmented, cut off from the British, almost encircled, and had only one narrow escape route to a single port and its nearby beaches.
